# Checkpoint 2 — Look at the Data

**Goal:** Understand what we're working with BEFORE writing any ML code.

**Rule:** Run one cell at a time. Read the output. Understand it. Then move to the next.

Every cell has a comment explaining what it does and WHY.

## Step 1 — Load the data

We use `pandas` to load the CSV file into a "DataFrame".
A DataFrame is like a spreadsheet in Python — rows and columns you can work with in code.

In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
# "../data/raw/..." means: go one folder up (from notebooks/), then into data/raw/
df = pd.read_csv("../data/raw/telco_churn.csv")

# Confirm it loaded correctly
print(f"Rows    : {df.shape[0]}")   # How many customers
print(f"Columns : {df.shape[1]}")   # How many things we know about each customer
print("\nLoad successful ✅")

## Step 2 — Look at one customer row

Before anything else, look at ONE real row.
This tells you what information we have about each customer.
Read every column name and value. Try to understand what it means.

In [ ]:
# .T means "transpose" — shows columns as rows so it's easier to read
# This is the FIRST customer in the dataset
df.iloc[0].to_frame(name="Value")

## Step 3 — What columns do we have?

Each column is one thing we know about a customer.
Some will help the model predict churn. Some might not.
Read each column name and think: "Does this make sense for predicting if someone will leave?"

In [ ]:
# Show each column name and what type of data it contains
# "object"  = text (like "Yes", "No", "Male", "Female")
# "int64"   = whole numbers (like 12, 45)
# "float64" = decimal numbers (like 29.85)
df.dtypes.to_frame(name="Data Type")

## Step 4 — How many customers churned?

`Churn` is our TARGET — the thing we're trying to predict.
- "Yes" = customer LEFT the company
- "No"  = customer STAYED

Before building any model, we need to know: is the data balanced?
If 99% of customers stayed, a model that always says "No" would be 99% accurate but completely useless!

In [ ]:
# Count how many customers churned vs stayed
churn_counts = df["Churn"].value_counts()
churn_pct    = df["Churn"].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    "Count"      : churn_counts,
    "Percentage" : churn_pct.round(1).astype(str) + "%"
})
print(summary)

# THINK ABOUT THIS:
# Is this balanced? (50/50 would be perfectly balanced)
# What happens if our model just guesses "No" every time?

In [ ]:
import matplotlib.pyplot as plt

# Draw a simple bar chart so we can SEE the imbalance visually
churn_counts.plot(kind="bar", color=["steelblue", "tomato"], edgecolor="black")

plt.title("Churned vs Stayed", fontsize=14)
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 5 — Are there any missing values?

Missing values are gaps in the data — a customer record where one field is blank.
ML models can't handle blanks — we need to fix them before training.
First, let's find out HOW MANY there are (if any).

In [ ]:
# Count missing values per column
missing = df.isnull().sum()

# Only show columns that actually have missing values
missing_cols = missing[missing > 0]

if len(missing_cols) == 0:
    print("No missing values found in any column ✅")
else:
    print("Columns with missing values:")
    print(missing_cols)

# BUT WAIT — some datasets hide missing values as empty strings " "
# Let's also check for that in TotalCharges (a common trap in this dataset)
print("\nChecking TotalCharges for hidden blanks...")
blank_total_charges = df[df["TotalCharges"].str.strip() == ""]
print(f"Rows with blank TotalCharges: {len(blank_total_charges)}")

## Step 6 — Your observations (fill this in yourself)

After running all the cells above, write your answers here.
THIS IS IMPORTANT — writing forces you to actually understand.

**Q1: How many customers are in the dataset?**
7,043

**Q2: How many columns (features) does each customer have?**
21

**Q3: What percentage of customers churned?**
26.5%

**Q4: Is the dataset balanced or imbalanced? What does that mean for our model?**
Imbalanced — 74% stayed, 26% churned. A model that always says "No churn" would be 74% accurate but would miss every churner. High accuracy alone is misleading. We need metrics like Recall that measure performance on the minority class.

**Q5: Did you find any missing or blank values? Which column?**
TotalCharges has 11 hidden blank rows stored as empty strings (not NaN), so isnull() misses them. We need to check with str.strip() == "".

**Q6: Look at the single customer row from Step 2. Name 3 columns you think will be USEFUL for predicting churn and explain why.**
- Contract: Month-to-month customers can leave anytime with no penalty — likely strongest predictor
- MonthlyCharges: Higher bills = more temptation to switch to a cheaper provider
- tenure: New customers haven't committed yet — more likely to leave early
(Note: customerID is NOT useful — it is just a random identifier with no relationship to churn)

---
Once you've answered these, paste them in the chat. Then we move to Checkpoint 3.